In [ ]:
!pip install --upgrade transformers tokenizers
!pip install --upgrade "transformers==4.28.0" "datasets" "scikit-learn"

In [1]:
#Imports
import torch
import torch.nn as nn
import math
import pandas as pd
import re
import os
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer

In [2]:
#Multi-Head Attention
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.d_k = dim // n_heads 
        self.dim = dim
        
    
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim) 
    
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        q = self.q_proj(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
     
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:

            scores = scores.masked_fill(mask == 0, -1e9)
            
        attn_weights = torch.softmax(scores, dim=-1)
    
        context = attn_weights @ v
        
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.dim)
        return self.out_proj(context)

#Feed-Forward Network
class FeedForward(nn.Module):
    def __init__(self, dim, ff_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, ff_dim),
            nn.GELU(), # BERT uses GELU
            nn.Dropout(dropout),
            nn.Linear(ff_dim, dim)
        )
    
    def forward(self, x):
        return self.net(x)

#Transformer Encoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, dim, n_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(dim, n_heads)
        self.ff = FeedForward(dim, ff_dim, dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
        
        
        self.report_shapes = False

    def forward(self, x, mask=None):
        attn_out = self.attn(x, x, x, mask)
        
        if self.report_shapes:
            print(f"[Report] Shape of Output (Multi-Head Attention): {attn_out.shape}")
            
        x = self.norm1(x + self.dropout(attn_out)) 
        
        #Feed-forward block
        ff_out = self.ff(x)

        if self.report_shapes:
            print(f"[Report] Shape of Output (Feed-Forward): {ff_out.shape}")

        x = self.norm2(x + self.dropout(ff_out)) 
        return x

#BERT Embeddings
class BertEmbeddings(nn.Module):
    def __init__(self, vocab_size, dim, max_len, n_segments):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.pos_embed = nn.Embedding(max_len, dim)
        self.seg_embed = nn.Embedding(n_segments, dim)
        
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(0.1)
        
        self.register_buffer("position_ids", torch.arange(max_len).expand((1, -1)))

    def forward(self, input_ids, segment_ids):
        seq_len = input_ids.size(1)
        tok_emb = self.tok_embed(input_ids)
        pos_emb = self.pos_embed(self.position_ids[:, :seq_len])
        seg_emb = self.seg_embed(segment_ids)
        embeddings = tok_emb + pos_emb + seg_emb
        return self.dropout(self.norm(embeddings))

#Main BERT Model Class
class MyBertModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config 
        
        self.embeddings = BertEmbeddings(
            config['vocab_size'], 
            config['dim'], 
            config['max_len'], 
            config['n_segments']
        )
        
        
        self.layers = nn.ModuleList(
            [EncoderLayer(config['dim'], config['n_heads'], config['ff_dim']) 
             for _ in range(config['n_layers'])]
        )
        
       
        self.pooler = nn.Sequential(
            nn.Linear(config['dim'], config['dim']),
            nn.Tanh()
        )
        
    
        self.classifier = nn.Linear(config['dim'], config['num_classes'])
    
    def forward(self, input_ids, segment_ids, attention_mask=None):
        
    
        if attention_mask is not None:
           
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
        else:
            mask = None
            
        #Get Embeddings
        x = self.embeddings(input_ids, segment_ids)
        if self.config.get('report_shapes', False): # Check config flag
            print(f"[Report] Shape of Token Embeddings: {x.shape}")
        
        #Pass through Encoder layers
        for layer in self.layers:
            x = layer(x, mask=mask)
        
        if self.config.get('report_shapes', False):
            print(f"[Report] Shape of Final Encoder Output (Layer {self.config['n_layers']}): {x.shape}")
        
        #Pooler
        cls_token_state = x[:, 0]
        pooled_output = self.pooler(cls_token_state)
        
        #Classifier
        logits = self.classifier(pooled_output)
        
        return logits
    
 
    def set_report_shapes(self, report=True):
        self.config['report_shapes'] = report
        for layer in self.layers:
            layer.report_shapes = report
            


print("--- [Task 1] Starting Test and Report ---")

#Define Model 

bert_config = {
    'dim': 768,
    'n_heads': 12,
    'n_layers': 2,
    'ff_dim': 768 * 4, 
    'num_classes': 2,   #For toxicity
    
    #Standard bert-base-uncased specs
    'vocab_size': 30522,
    'max_len': 512,
    'n_segments': 2,
}

#Initialize Tokenizer
print("Loading 'bert-base-uncased' tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

#Define and Tokenize Sample Sentence
sentence = "This is a sample sentence with more than 10 words, which we will use to test our BERT model."
print(f"Sample Sentence: '{sentence}'")

#Tokenize
inputs = tokenizer(
    sentence,
    return_tensors='pt',
    max_length=64,
    padding='max_length',
    truncation=True
)

input_ids = inputs['input_ids']
attn_mask = inputs['attention_mask']

seg_ids = torch.zeros_like(input_ids) 

print(f"Tokenized 'input_ids' shape: {input_ids.shape}")

#Instantiate and Test Model
print("\nInstantiating custom MyBertModel")
model = MyBertModel(bert_config)
model.eval() 
model.set_report_shapes(True)


print("\n--- [Report] Shapes of Embeddings and Outputs (from forward pass) ---")
with torch.no_grad():
    output_probs = model(input_ids, seg_ids, attn_mask)

print(f"[Report] Shape of Output Probabilities (Classifier): {output_probs.shape}")

#Report: Shape of each parameter
print("\n--- [Report] Shape of Each Parameter ---")
total_params = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: \t{param.shape}")
        total_params += param.numel()

print(f"\nTotal Model Parameters: {total_params / 1_000_000:.2f}M")
print("(Note: A full BERT-base has ~110M params; ours is smaller with 2 layers)")
print("\n--- [Task 1] Test and Report Complete ---")

--- [Task 1] Starting Test and Report ---
Loading 'bert-base-uncased' tokenizer...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Sample Sentence: 'This is a sample sentence with more than 10 words, which we will use to test our BERT model.'
Tokenized 'input_ids' shape: torch.Size([1, 64])

Instantiating custom MyBertModel

--- [Report] Shapes of Embeddings and Outputs (from forward pass) ---
[Report] Shape of Token Embeddings: torch.Size([1, 64, 768])
[Report] Shape of Output (Multi-Head Attention): torch.Size([1, 64, 768])
[Report] Shape of Output (Feed-Forward): torch.Size([1, 64, 768])
[Report] Shape of Output (Multi-Head Attention): torch.Size([1, 64, 768])
[Report] Shape of Output (Feed-Forward): torch.Size([1, 64, 768])
[Report] Shape of Final Encoder Output (Layer 2): torch.Size([1, 64, 768])
[Report] Shape of Output Probabilities (Classifier): torch.Size([1, 2])

--- [Report] Shape of Each Parameter ---
embeddings.tok_embed.weight: 	torch.Size([30522, 768])
embeddings.pos_embed.weight: 	torch.Size([512, 768])
embeddings.seg_embed.weight: 	torch.Size([2, 768])
embeddings.norm.weight: 	torch.Size([768])
em

In [3]:
FOLDER_PATH = "/kaggle/input/all-samples/all_samples (also includes non multimodal)"

FILES_TO_COMBINE = [
    'all_test_public.tsv',
    'all_train.tsv',
    'all_validate.tsv'
]
COLUMNS_TO_KEEP = ['clean_title', '2_way_label', 'id']
LABEL_COL = '2_way_label'
TEXT_COL = 'clean_title'

MIN_SAMPLES = 2500 
TEST_SIZE = 0.2
SEED = 42 

print("--- [Task 2] Starting Fakeddit Preprocessing (Corrected) ---")

#Step 1: Load and Combine Files
all_dfs = []
print(f"Loading files from: {FOLDER_PATH}")

try:
    for f_name in FILES_TO_COMBINE:
        #Create the full path to the file
        file_path = os.path.join(FOLDER_PATH, f_name)
        print(f"  - Loading {f_name}...")
        
        #Load the .tsv file
        df = pd.read_csv(file_path, sep='\t')
        all_dfs.append(df)

except FileNotFoundError as e:
    print(f"\nError: FILE NOT FOUND.")
    print(f"Could not find file at: {e.filename}")
    print("Please make sure 'FOLDER_PATH' is correct and contains all 3 files.")
    raise
except Exception as e:
    print(f"An error occurred: {e}")
    raise

#Combine all loaded dataframes into one
df_original = pd.concat(all_dfs, ignore_index=True)
original_count = len(df_original)
print(f"Loaded and combined {original_count:,} total posts.")

#Filter to keep only the required columns
if not all(col in df_original.columns for col in COLUMNS_TO_KEEP):
    print(f"Error: Missing required columns.")
    print(f"Required: {COLUMNS_TO_KEEP}")
    print(f"Found: {df_original.columns.tolist()}")
    raise ValueError("Missing required columns from combined files.")

df_filtered = df_original[COLUMNS_TO_KEEP].copy()


df_filtered.dropna(subset=[TEXT_COL], inplace=True)
df_filtered.drop_duplicates(subset=[TEXT_COL], inplace=True)
print(f"Cleaned data: {len(df_filtered):,} posts remaining after removing duplicates/NaNs.")

#Step 3: Balance the Dataset
print(f"Step 3: Balancing to {MIN_SAMPLES} real and {MIN_SAMPLES} fake...")

#2_way_label: 0 = Real, 1 = Fake
real_count = (df_filtered[LABEL_COL] == 0).sum()
fake_count = (df_filtered[LABEL_COL] == 1).sum()
print(f"        (Cleaned counts: {real_count} real, {fake_count} fake)")

#Check if we have enough data
if fake_count < MIN_SAMPLES or real_count < MIN_SAMPLES:
    print(f"Warning: Not enough samples. Using {min(fake_count, real_count)} instead.")
    MIN_SAMPLES = min(fake_count, real_count)

#Separate the two classes
real_df = df_filtered[df_filtered[LABEL_COL] == 0]
fake_df = df_filtered[df_filtered[LABEL_COL] == 1]

#Sample from each
real_sample = real_df.sample(n=MIN_SAMPLES, random_state=SEED)
fake_sample = fake_df.sample(n=MIN_SAMPLES, random_state=SEED)

#Combine and shuffle
df_balanced = pd.concat([real_sample, fake_sample])
df_balanced = df_balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)
count_balanced = len(df_balanced)
print(f"        Created balanced set of {count_balanced} rows.")

#Train-Test Split (80/20)
print(f"Step 4: Splitting into 80% train / 20% test...")

train_df, test_df = train_test_split(
    df_balanced,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df_balanced[LABEL_COL] 
)

#Report Statistics (as required)
print("\n--- [Task 2] Final Statistics Report ---")
print(f"1. Number of posts in the original dataset (combined): {original_count}")
print(f"2. Number of fake and non-fake posts in the balanced dataset built:")
print(f"   - Fake ({LABEL_COL}=1):     {(df_balanced[LABEL_COL] == 1).sum()}")
print(f"   - Non-Fake ({LABEL_COL}=0): {(df_balanced[LABEL_COL] == 0).sum()}")
print(f"   - Total:             {count_balanced}")
print(f"3. Distribution of posts in train and test sets:")
print(f"   - Train Set: {len(train_df)} rows")
print(f"   - Test Set:  {len(test_df)} rows")
print(f"   - Train 'fake' count: {(train_df[LABEL_COL] == 1).sum()}")
print(f"   - Test 'fake' count:  {(test_df[LABEL_COL] == 1).sum()}")
print("\n--- [Task 2] Preprocessing Complete ---")

--- [Task 2] Starting Fakeddit Preprocessing (Corrected) ---
Loading files from: /kaggle/input/all-samples/all_samples (also includes non multimodal)
  - Loading all_test_public.tsv...
  - Loading all_train.tsv...
  - Loading all_validate.tsv...
Loaded and combined 1,063,106 total posts.
Cleaned data: 863,876 posts remaining after removing duplicates/NaNs.
Step 3: Balancing to 2500 real and 2500 fake...
        (Cleaned counts: 407086 real, 456790 fake)
        Created balanced set of 5000 rows.
Step 4: Splitting into 80% train / 20% test...

--- [Task 2] Final Statistics Report ---
1. Number of posts in the original dataset (combined): 1063106
2. Number of fake and non-fake posts in the balanced dataset built:
   - Fake (2_way_label=1):     2500
   - Non-Fake (2_way_label=0): 2500
   - Total:             5000
3. Distribution of posts in train and test sets:
   - Train Set: 4000 rows
   - Test Set:  1000 rows
   - Train 'fake' count: 2000
   - Test 'fake' count:  500

--- [Task 2] Prep

In [4]:
import torch
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
import numpy as np

try:
    _ = train_df
    _ = test_df
    print("Found 'train_df' and 'test_df'. Proceeding with Task 3.")
except NameError:
    print("Error: 'train_df' and 'test_df' not found.")
    print("Please make sure your Task 2 cell has run successfully before this.")
    raise

print("--- [Task 3] Starting Model Fine-Tuning ---")

#Define Model & Tokenizer
MODEL_NAME = 'bert-base-uncased'
TEXT_COL = 'clean_title'
LABEL_COL = '2_way_label' 

print(f"Loading tokenizer and model for '{MODEL_NAME}'...")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

#Define Hyperparameters (for the report)
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5

hyperparameters = {
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "optimizer": "AdamW (default)",
    "task": "Fakeddit 2-way classification"
}

print("Hyperparameters:")
for key, val in hyperparameters.items():
    print(f"  - {key}: {val}")

#Prepare Data for Transformers
print("Converting pandas DataFrames to Hugging Face Datasets...")
train_dataset = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: 'label'}))
test_dataset = Dataset.from_pandas(test_df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: 'label'}))

#Tokenization function
def tokenize_data(examples):
    return tokenizer(
        examples[TEXT_COL], 
        padding="max_length", 
        truncation=True
    )

print("Tokenizing datasets (this may take a minute)...")
train_dataset = train_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])


#Define Metrics Function (for the report)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)
    cm = confusion_matrix(labels, preds)
    
    print("\nTest Set Confusion Matrix:\n", cm)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

#Set up Trainer

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    

    evaluation_strategy="epoch",  
    save_strategy="epoch",        
    logging_steps=100,            
  

    load_best_model_at_end=True,
    report_to="none"             
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Train the Model
print("\n Starting Model Training (make sure GPU is on)")
trainer.train() 
print("--- Model Training Complete ---")


print("\n [Task 3] Final Evaluation on Test Set")
final_metrics = trainer.evaluate()

print("\n [Task 3] Final Results Report")

print("\n Hyperparameters Used")
for key, val in hyperparameters.items():
    print(f"  - {key}: {val}")

print("\nTest Set Metrics")
print(f"  - Accuracy:  {final_metrics['eval_accuracy']:.4f}")
print(f"  - Precision: {final_metrics['eval_precision']:.4f}")
print(f"  - Recall:    {final_metrics['eval_recall']:.4f}")
print(f"  - F1-score:  {final_metrics['eval_f1']:.4f}")
print("\n(Confusion Matrix was printed during evaluation)")

2025-11-05 13:22:32.878658: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762348953.287495      99 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762348953.419886      99 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Found 'train_df' and 'test_df'. Proceeding with Task 3.
--- [Task 3] Starting Model Fine-Tuning ---
Loading tokenizer and model for 'bert-base-uncased'...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i

...


Test Set Confusion Matrix:
 [[324 176]
 [ 72 428]]


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



Test Set Confusion Matrix:
 [[421  79]
 [136 364]]
--- Model Training Complete ---

 [Task 3] Final Evaluation on Test Set


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



Test Set Confusion Matrix:
 [[421  79]
 [136 364]]

 [Task 3] Final Results Report

 Hyperparameters Used
  - model: bert-base-uncased
  - epochs: 3
  - batch_size: 16
  - learning_rate: 2e-05
  - optimizer: AdamW (default)
  - task: Fakeddit 2-way classification

Test Set Metrics
  - Accuracy:  0.7850
  - Precision: 0.8217
  - Recall:    0.7280
  - F1-score:  0.7720

(Confusion Matrix was printed during evaluation)


In [6]:
#Task 2 Bonus

COMMENTS_FILE_PATH = "/kaggle/input/comments/all_comments.tsv/all_comments.tsv" 
try:
    _ = df_original
    _ = df_balanced
except NameError:
    print("Error: 'df_original' and 'df_balanced' not found.")
    print("Please re-run your main Task 2 cell first!")
    raise

print("[Task 2 Bonus] Starting")

print(f"Loading comments from '{COMMENTS_FILE_PATH}'...")
try:

    df_comments = pd.read_csv(
        COMMENTS_FILE_PATH, 
        sep='\t', 
        usecols=['submission_id']
    )
    print(f"Loaded {len(df_comments):,} total comments.")
except FileNotFoundError:
    print(f"ERROR: File not found at '{COMMENTS_FILE_PATH}'")
    print("Please fix the path variable at the top of the script.")
    raise
except Exception as e:
    print(f"Error loading data: {e}")
    raise


print("Counting comments per post...")
comment_counts = df_comments['submission_id'].value_counts().reset_index()
comment_counts.columns = ['id', 'comment_count'] 
print(f"Found comments for {len(comment_counts):,} unique posts.")


df_original_with_comments = pd.merge(df_original, comment_counts, on='id', how='left')
df_balanced_with_comments = pd.merge(df_balanced, comment_counts, on='id', how='left')

df_original_with_comments['comment_count'] = df_original_with_comments['comment_count'].fillna(0).astype(int)
df_balanced_with_comments['comment_count'] = df_balanced_with_comments['comment_count'].fillna(0).astype(int)

print("Merged comment counts with post dataframes.")

print("\n [Task 2 Bonus] Statistics Report")

posts_with_comments = (df_original_with_comments['comment_count'] > 0).sum()
print(f"i.   Number of posts with at least one comment (original dataset): {posts_with_comments:,}")

print("\nii.  Comment stats for fake/non-fake (entire dataset):")
stats_original = df_original_with_comments.groupby('2_way_label')['comment_count'].agg(['mean', 'std'])
print(stats_original)

print("\niii. Comment stats for fake/non-fake (balanced dataset):")
stats_balanced = df_balanced_with_comments.groupby('2_way_label')['comment_count'].agg(['mean', 'std'])
print(stats_balanced)


[Task 2 Bonus] Starting
Loading comments from '/kaggle/input/comments/all_comments.tsv/all_comments.tsv'...


/tmp/ipykernel_99/2437892036.py:17: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_comments = pd.read_csv(


Loaded 10,697,533 total comments.
Counting comments per post...
Found comments for 618,948 unique posts.
Merged comment counts with post dataframes.

 [Task 2 Bonus] Statistics Report
i.   Number of posts with at least one comment (original dataset): 584,098

ii.  Comment stats for fake/non-fake (entire dataset):
                  mean        std
2_way_label                      
0             3.845314  18.067710
1            14.885789  56.057783

iii. Comment stats for fake/non-fake (balanced dataset):
                mean        std
2_way_label                    
0             4.6060  17.696622
1            15.3148  56.225149
